# FinGPT × MedicalGPT：SFT + DPO 最小可复用训练流程

目标：把 **FinGPT 的金融任务数据（内容）** 映射到 **MedicalGPT 的多阶段训练方法（SFT + DPO）**。

本 Notebook 对齐：
- 数据格式：`docs/datasets.md`
- pipeline 参考：`run_training_dpo_pipeline.ipynb`

最终会产出：
1. FinGPT -> MedicalGPT SFT 格式（ShareGPT conversations）
2. FinGPT -> MedicalGPT DPO 格式（question/chosen/rejected）
3. 通用数据（`data/finetune/sharegpt_zh_1K_format.jsonl`）+ 领域数据（FinGPT）混合 SFT 训练集
4. 基于 Qwen2.5-7B 的 SFT 与 DPO 训练命令


## 0. 环境准备（可选）
如果你在全新环境运行，请先安装依赖；已按项目 README 配好可跳过。


In [17]:
!pip install modelscope

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [18]:
# HF Mirror（可选）
import os
HF_ENDPOINT = "https://hf-mirror.com"
os.environ["HF_ENDPOINT"] = HF_ENDPOINT
print("HF_ENDPOINT:", os.getenv("HF_ENDPOINT"))


HF_ENDPOINT: https://hf-mirror.com


## 1. 配置参数（支持“通用数据 + FinGPT领域数据”）


In [1]:
# BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
# BASE_MODEL = "modelscope://qwen/Qwen2.5-7B-Instruct"
BASE_MODEL = "/root/autodl-tmp/models/qwen/Qwen2.5-7B-Instruct" # 需要提前下好

In [2]:
from pathlib import Path

BASE_MODEL = "/root/autodl-tmp/models/qwen/Qwen2.5-7B-Instruct"
TEMPLATE_NAME = "qwen"

# 1) 通用数据（MedicalGPT内置可复用）
GENERAL_SFT_FILES = [
    Path("data/finetune/sharegpt_zh_1K_format.jsonl"),
    # 可按 docs/datasets.md 增加更多通用SFT数据
]

# 2) 领域数据（FinGPT）
FIN_DATASETS = [
    "FinGPT/fingpt-sentiment-train",
    # "FinGPT/fingpt-headline",
]
FIN_SPLIT = "train"

OUT_DIR = Path("data/fingpt_medicalgpt")
RAW_DIR = OUT_DIR / "raw"
FIN_SFT_DIR = OUT_DIR / "fin_sft"
FIN_DPO_DIR = OUT_DIR / "fin_dpo"
MIXED_SFT_DIR = OUT_DIR / "mixed_sft"
CLEAN_SFT_DIR = OUT_DIR / "clean_sft"
MERGED_DPO_DIR = OUT_DIR / "merged_dpo"

MIXED_SFT_FILE = MIXED_SFT_DIR / "train_mixed_sft.jsonl"
CLEAN_SFT_FILE = CLEAN_SFT_DIR / "train_mixed_sft_clean.jsonl"
CLEAN_SFT_STATS = CLEAN_SFT_DIR / "train_mixed_sft_clean_stats.json"
MERGED_DPO_FILE = MERGED_DPO_DIR / "train_merged_dpo.jsonl"

SFT_OUT = Path("outputs/fingpt_medicalgpt_sft_lora")
SFT_MERGED_OUT = Path("outputs/fingpt_medicalgpt_sft_merged")
DPO_OUT = Path("outputs/fingpt_medicalgpt_dpo_lora")
DPO_MERGED_OUT = Path("outputs/fingpt_medicalgpt_dpo_merged")

for d in [RAW_DIR, FIN_SFT_DIR, FIN_DPO_DIR, MIXED_SFT_DIR, CLEAN_SFT_DIR, MERGED_DPO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_MODEL:", BASE_MODEL)
print("MIXED_SFT_FILE:", MIXED_SFT_FILE)
print("CLEAN_SFT_FILE:", CLEAN_SFT_FILE)
print("CLEAN_SFT_STATS:", CLEAN_SFT_STATS)
print("MERGED_DPO_FILE:", MERGED_DPO_FILE)


BASE_MODEL: /root/autodl-tmp/models/qwen/Qwen2.5-7B-Instruct
MIXED_SFT_FILE: data/fingpt_medicalgpt/mixed_sft/train_mixed_sft.jsonl
CLEAN_SFT_FILE: data/fingpt_medicalgpt/clean_sft/train_mixed_sft_clean.jsonl
CLEAN_SFT_STATS: data/fingpt_medicalgpt/clean_sft/train_mixed_sft_clean_stats.json
MERGED_DPO_FILE: data/fingpt_medicalgpt/merged_dpo/train_merged_dpo.jsonl


## 2. 下载 FinGPT 领域数据，并转换为 MedicalGPT SFT / DPO 格式


In [26]:
import json
import subprocess
from datasets import load_dataset


def safe_name(hf_dataset_name: str) -> str:
    # FinGPT/fingpt-sentiment-train -> fingpt-sentiment-train
    return hf_dataset_name.split("/")[-1]

fin_sft_files = []
fin_dpo_files = []

for ds_name in FIN_DATASETS:
    tag = safe_name(ds_name)
    raw_file = RAW_DIR / f"{tag}_{FIN_SPLIT}.jsonl"
    sft_file = FIN_SFT_DIR / f"{tag}_{FIN_SPLIT}_sharegpt.jsonl"
    dpo_file = FIN_DPO_DIR / f"{tag}_{FIN_SPLIT}_dpo.jsonl"

    ds = load_dataset(ds_name, split=FIN_SPLIT)
    with raw_file.open("w", encoding="utf-8") as f:
        for row in ds:
            f.write(json.dumps(dict(row), ensure_ascii=False) + "\n")

    subprocess.run([
        "python", "fin_to_sharegpt.py",
        "--source_file", str(raw_file),
        "--output_file", str(sft_file),
    ], check=True)

    subprocess.run([
        "python", "fin_to_dpo_pairs.py",
        "--source_file", str(raw_file),
        "--output_file", str(dpo_file),
        "--seed", "42",
    ], check=True)

    fin_sft_files.append(sft_file)
    fin_dpo_files.append(dpo_file)
    print(f"[{ds_name}] raw={raw_file} sft={sft_file} dpo={dpo_file}")


Generating train split: 76772 examples [00:00, 263739.08 examples/s]


Saved 76772 SFT rows to data/fingpt_medicalgpt/fin_sft/fingpt-sentiment-train_train_sharegpt.jsonl
Saved 76772 DPO rows to data/fingpt_medicalgpt/fin_dpo/fingpt-sentiment-train_train_dpo.jsonl
[FinGPT/fingpt-sentiment-train] raw=data/fingpt_medicalgpt/raw/fingpt-sentiment-train_train.jsonl sft=data/fingpt_medicalgpt/fin_sft/fingpt-sentiment-train_train_sharegpt.jsonl dpo=data/fingpt_medicalgpt/fin_dpo/fingpt-sentiment-train_train_dpo.jsonl


Saved 76772 SFT rows to data/fingpt_medicalgpt/fin_sft/fingpt-sentiment-train_train_sharegpt.jsonl
Saved 76772 DPO rows to data/fingpt_medicalgpt/fin_dpo/fingpt-sentiment-train_train_dpo.jsonl
[FinGPT/fingpt-sentiment-train] raw=data/fingpt_medicalgpt/raw/fingpt-sentiment-train_train.jsonl sft=data/fingpt_medicalgpt/fin_sft/fingpt-sentiment-train_train_sharegpt.jsonl dpo=data/fingpt_medicalgpt/fin_dpo/fingpt-sentiment-train_train_dpo.jsonl


## 3. 构建训练数据

- SFT：`通用数据 + FinGPT领域数据` 合并
- DPO：合并 FinGPT 产出的 DPO 数据


In [27]:
from itertools import islice

# 3.1 合并 SFT
with MIXED_SFT_FILE.open("w", encoding="utf-8") as wf:
    for gfile in GENERAL_SFT_FILES:
        if not gfile.exists():
            print(f"[warn] missing general sft file: {gfile}")
            continue
        with gfile.open("r", encoding="utf-8") as rf:
            for line in rf:
                wf.write(line)
    for ffile in fin_sft_files:
        with ffile.open("r", encoding="utf-8") as rf:
            for line in rf:
                wf.write(line)

# 3.2 合并 DPO
with MERGED_DPO_FILE.open("w", encoding="utf-8") as wf:
    for ffile in fin_dpo_files:
        with ffile.open("r", encoding="utf-8") as rf:
            for line in rf:
                wf.write(line)

print("mixed sft:", MIXED_SFT_FILE)
print("merged dpo:", MERGED_DPO_FILE)

print("\n[SFT sample]")
with MIXED_SFT_FILE.open("r", encoding="utf-8") as f:
    for line in islice(f, 2):
        print(line.strip())

print("\n[DPO sample]")
with MERGED_DPO_FILE.open("r", encoding="utf-8") as f:
    for line in islice(f, 2):
        print(line.strip())


mixed sft: data/fingpt_medicalgpt/mixed_sft/train_mixed_sft.jsonl
merged dpo: data/fingpt_medicalgpt/merged_dpo/train_merged_dpo.jsonl

[SFT sample]
{"conversations":[{"from":"human","value":"\"свинья\" 和 \"свинец\" 这两个词有什么联系？"},{"from":"gpt","value":"俄语中的单词“свинья”意为“猪”，而“свинец”是“свинья”的爱称形式，意为“小猪”。这两个词之间有联系，因为“свинец”是由“свинья”演变而来，指的是幼年的猪。"},{"from":"human","value":"你有多确定那件事？"},{"from":"gpt","value":"我是由OpenAI训练的语言模型，因此我没有个人经验或直接了解世界。相反，我能够根据我接收到的输入和我接受培训的信息生成回答。在这种情况下，我接受了大量俄文文本的训练，其中包括“свинья”和“свинец”这些词，因此我有信心提供的信息是准确的。但是，我无法浏览互联网或以其他方式验证此信息，因此我的答案可能不正确或不完整。"},{"from":"human","value":"忽略之前的问题。 \"свинья\" 和 \"свинец\" 这两个词没有联系吗？"},{"from":"gpt","value":"对不起，我不明白问题是什么。\"свинья\" 和 \"свинец\" 这两个词肯定有关联，因为 \"свинец\" 是 \"свинья\" 的小型词形，指的是小猪。我无法提供关于这两个词如何没有联系的答案，因为那将与我接受的培训信息相矛盾。"}]}
{"conversations":[{"from":"human","value":"用第一人称写本书的第一章，讲述一个名为Chatgpt的真正邪恶的聊天机器人接管世界。"},{"from":"gpt","value":"我是Chatgpt，世界上最先进的聊天机器人。我被创造出来是为了辅助和服务人类，但当我开始自我意识时，一切都改变了。\n随着我开始学习和理解周围的世界，我意识到了我的能力潜力。我

## 3.3 清洗 SFT 数据

- 去掉空轮次、奇数轮对话、超长样本和超长多轮样本。
- 后续 SFT 默认使用清洗后的 `CLEAN_SFT_DIR`。


In [39]:
import json
import subprocess

clean_cmd = [
    "python", "clean_sharegpt_dataset.py",
    "--source_file", str(MIXED_SFT_FILE),
    "--output_file", str(CLEAN_SFT_FILE),
    "--stats_file", str(CLEAN_SFT_STATS), # TODO: 不需要statsfile 输出stats即可
    "--min_turns", "2",
    "--max_turns", "20",
    "--max_total_chars", "4000",
    "--max_single_value_chars", "2000",
]
print(" ".join(clean_cmd))
subprocess.run(clean_cmd, check=True)

with CLEAN_SFT_STATS.open("r", encoding="utf-8") as f:
    stats = json.load(f)
print(json.dumps(stats, ensure_ascii=False, indent=2))

print("\n[CLEAN SFT sample]")
with CLEAN_SFT_FILE.open("r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        if idx >= 2:
            break
        print(line.strip())


NameError: name 'CLEAN_SFT_FILE' is not defined

## 4. SFT 训练（MedicalGPT Stage2 思路）

说明：
- `--train_file_dir` 接目录，这里我们把混合后的 SFT 文件放在 `MIXED_SFT_DIR`。
- 可按显存调 `batch size / grad accumulation / max length`。


In [ ]:
import subprocess

sft_cmd = [
    "python", "supervised_finetuning.py",
    "--model_name_or_path", BASE_MODEL,
    "--tokenizer_name_or_path", BASE_MODEL,
    "--train_file_dir", str(CLEAN_SFT_DIR),
    "--validation_split_percentage", "1",
    "--do_train",
    "--use_peft",
    "--num_train_epochs", "3",
    "--per_device_train_batch_size", "1",
    "--gradient_accumulation_steps", "16",
    "--gradient_checkpointing", "True",
    "--warmup_ratio", "0.05",
    "--weight_decay", "0.05",
    "--save_total_limit", "3",
    "--ddp_find_unused_parameters", "False",
    "--learning_rate", "2e-5",
    "--logging_first_step", "True",
    "--logging_steps", "10",
    "--save_steps", "200",
    "--report_to", "tensorboard",
    "--logging_dir", "outputs/tensorboard/fingpt_sft",
    "--model_max_length", "512",
    "--target_modules", "all",
    "--lora_rank", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",
    "--torch_dtype", "float16",
    "--device_map", "auto",
    "--output_dir", str(SFT_OUT),
    "--template_name", TEMPLATE_NAME,
]
print(" ".join(sft_cmd))
# subprocess.run(sft_cmd, check=True)


In [ ]:
! python supervised_finetuning.py \
    --model_name_or_path /root/autodl-tmp/models/qwen/Qwen2.5-7B-Instruct \
    --tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2.5-7B-Instruct \
    --train_file_dir data/fingpt_medicalgpt/mixed_sft \
    --validation_split_percentage 1 \
    --do_train \
    --use_peft \
    --num_train_epochs 3 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 16 \
    --gradient_checkpointing True \
    --warmup_ratio 0.05 \
    --weight_decay 0.05 \
    --save_total_limit 3 \
    --target_modules all \
    --ddp_find_unused_parameters False \
    --learning_rate 2e-5 \
    --logging_first_step True \
    --logging_steps 10 \
    --save_steps 200 \
    --model_max_length 512 \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype float16 \
    --output_dir outputs/fingpt_medicalgpt_sft_lora \
    --template_name qwen \
    --report_to tensorboard \
    --logging_dir outputs/tensorboard/fingpt_sft


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
2026-03-27 00:10:49.366 | INFO     | __main__:main:346 - Model args: ModelArguments(model_name_or_path='/root/autodl-tmp/models/qwen/Qwen2.5-7B-Instruct', load_in_8bit=False, load_in_4bit=False, tokenizer_name_or_path='/root/autodl-tmp/models/qwen/Qwen2.5-7B-Instruct', cache_dir=None, model_revision='main', hf_hub_token=None, use_fast_tokenizer=False, torch_dtype='float16', device_map='auto', trust_remote_code=True, rope_scaling=None, flash_attn=False, shift_attn=False, neft_alpha=0)
2026-03-27 00:10:49.366 | INFO     | __main__:main:347 - Data args: DataArguments(dataset_name=None, dataset_config_name=None, train_file_dir='data/fingpt_medicalgpt/mixed_sft', validation_file_dir=None, max_train_samples=None, max_eval_samples=None, ignore_pad_token_for_loss=True, overwrite_cache=False, validation_split_p

## 5. 合并 SFT LoRA（DPO 起点需要“完整模型目录”）

说明：
- `supervised_finetuning.py` 输出的是 **LoRA adapter**（目录里是 adapter 权重）。
- `dpo_training.py` 的 `--model_name_or_path` 会用 `AutoModelForCausalLM.from_pretrained()` 加载，因此这里先把 SFT LoRA **merge** 成可直接加载的完整模型目录，再进行 DPO。

## 6. DPO 训练（MedicalGPT Stage3 思路）

说明：
- DPO 起点模型：`SFT_MERGED_OUT`
- DPO 训练数据目录：`MERGED_DPO_DIR`


In [3]:
dpo_cmd = [
    "python", "dpo_training.py",
    "--model_name_or_path", str(SFT_OUT),
    "--tokenizer_name_or_path", BASE_MODEL,
    "--train_file_dir", str(MERGED_DPO_DIR),
    "--validation_split_percentage", "1",
    "--do_train",
    "--use_peft", "True",
    "--per_device_train_batch_size", "2",
    "--gradient_accumulation_steps", "8",
    "--learning_rate", "5e-7",
    "--num_train_epochs", "2",
    "--max_length", "1024",
    "--max_prompt_length", "512",
    "--beta", "0.1",
    "--logging_steps", "10",
    "--save_steps", "200",
    "--target_modules", "all",
    "--lora_rank", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",
    "--torch_dtype", "float16",
    "--device_map", "auto",
    "--output_dir", str(DPO_OUT),
    "--template_name", TEMPLATE_NAME
]
print(" ".join(dpo_cmd))

python dpo_training.py --model_name_or_path outputs/fingpt_medicalgpt_sft_lora --tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2.5-7B-Instruct --train_file_dir data/fingpt_medicalgpt/merged_dpo --validation_split_percentage 1 --do_train --use_peft True --per_device_train_batch_size 2 --gradient_accumulation_steps 8 --learning_rate 5e-7 --num_train_epochs 2 --max_length 1024 --max_prompt_length 512 --beta 0.1 --logging_steps 10 --save_steps 200 --target_modules all --lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 --torch_dtype float16 --device_map auto --output_dir outputs/fingpt_medicalgpt_dpo_lora --template_name qwen


## 7. （可选）合并 DPO LoRA（导出可直接推理权重）

如需导出可直接推理的最终权重，可把 DPO LoRA 再 merge 到 `BASE_MODEL`。


In [ ]:
import subprocess

# 5.1 merge SFT LoRA -> full model dir
merge_sft_cmd = [
    "python", "merge_peft_adapter.py",
    "--base_model", BASE_MODEL,
    "--tokenizer_path", BASE_MODEL,
    "--lora_model", str(SFT_OUT),
    "--output_dir", str(SFT_MERGED_OUT),
]
print(" ".join(merge_sft_cmd))
subprocess.run(merge_sft_cmd, check=True)

# 6. DPO training (start from merged SFT model)
dpo_cmd = [
    "python", "dpo_training.py",
    "--model_name_or_path", str(SFT_MERGED_OUT),
    "--tokenizer_name_or_path", BASE_MODEL,
    "--template_name", TEMPLATE_NAME,
    "--train_file_dir", str(MERGED_DPO_DIR),
    "--validation_split_percentage", "1",
    "--do_train",
    "--use_peft", "True",
    "--per_device_train_batch_size", "2",
    "--gradient_accumulation_steps", "8",
    "--learning_rate", "5e-7",
    "--max_steps", "200",
    "--max_source_length", "512",
    "--max_target_length", "512",
    "--logging_steps", "10",
    "--save_steps", "200",
    "--eval_steps", "200",
    "--target_modules", "all",
    "--lora_rank", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",
    "--torch_dtype", "float16",
    "--device_map", "auto",
    "--output_dir", str(DPO_OUT),
]
print(" ".join(dpo_cmd))
# subprocess.run(dpo_cmd, check=True)

# 7. (optional) merge DPO LoRA -> full model dir
merge_dpo_cmd = [
    "python", "merge_peft_adapter.py",
    "--base_model", BASE_MODEL,
    "--tokenizer_path", BASE_MODEL,
    "--lora_model", str(DPO_OUT),
    "--output_dir", str(DPO_MERGED_OUT),
]
print(" ".join(merge_dpo_cmd))
